In [ ]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.6 MB/s eta 0:00:00


In [ ]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry
import psycopg2
from psycopg2.extras import execute_values
from datetime import datetime, date, timedelta

# -----------------------------------------------------------
# Setup Open-Meteo client
# -----------------------------------------------------------
cache_session = requests_cache.CachedSession('.cache', expire_after=0)  # no caching
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

today = date.today()
seven_days = today + timedelta(days=6) # Changed from 7 to 6 to stay within API's end_date range

# -----------------------------------------------------------
# WEATHER + SOIL API
# -----------------------------------------------------------
weather_url = "https://api.open-meteo.com/v1/forecast"
weather_params = {
    "latitude": 38.2527,
    "longitude": -85.7585,
    "hourly": [
        "temperature_2m", "relative_humidity_2m", "precipitation_probability",
        "precipitation", "wind_speed_10m", "soil_temperature_0cm",
        "soil_moisture_0_to_1cm"
    ],
    "timezone": "auto",
    "wind_speed_unit": "mph",
    "temperature_unit": "fahrenheit",
    "precipitation_unit": "inch",
    "start_date": today.isoformat(),
    "end_date": seven_days.isoformat(),
}

weather_response = openmeteo.weather_api(weather_url, params=weather_params)[0]
hourly = weather_response.Hourly()

weather_dates = pd.date_range(
    start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
    end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
    freq=pd.Timedelta(seconds=hourly.Interval()),
    inclusive="left"
).tz_convert(weather_response.Timezone().decode()).tz_localize(None)

weather_df = pd.DataFrame({
    "date": weather_dates,
    "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
    "relative_humidity_2m": hourly.Variables(1).ValuesAsNumpy(),
    "precipitation_probability": hourly.Variables(2).ValuesAsNumpy(),
    "precipitation": hourly.Variables(3).ValuesAsNumpy(),
    "wind_speed_10m": hourly.Variables(4).ValuesAsNumpy(),
    "soil_temperature_0cm": hourly.Variables(5).ValuesAsNumpy(),
    "soil_moisture_0_to_1cm": hourly.Variables(6).ValuesAsNumpy(),
})

# -----------------------------------------------------------
# AIR QUALITY + POLLEN API
# -----------------------------------------------------------
aq_url = "https://air-quality-api.open-meteo.com/v1/air-quality"
aq_params = {
    "latitude": 38.2527,
    "longitude": -85.7585,
    "hourly": [
        "pm10", "pm2_5", "carbon_monoxide", "ozone", "nitrogen_dioxide",
        "sulphur_dioxide", "us_aqi", "us_aqi_pm2_5", "us_aqi_pm10",
        "us_aqi_nitrogen_dioxide", "us_aqi_carbon_monoxide",
        "us_aqi_ozone", "us_aqi_sulphur_dioxide",
        "grass_pollen", "ragweed_pollen", "olive_pollen",
        "mugwort_pollen", "birch_pollen", "alder_pollen"
    ],
    "timezone": "auto",
    "domains": "cams_global",
    "start_date": today.isoformat(),
    "end_date": seven_days.isoformat(),
}

aq_response = openmeteo.weather_api(aq_url, params=aq_params)[0]
aq_hourly = aq_response.Hourly()

aq_dates = pd.date_range(
    start=pd.to_datetime(aq_hourly.Time(), unit="s", utc=True),
    end=pd.to_datetime(aq_hourly.TimeEnd(), unit="s", utc=True),
    freq=pd.Timedelta(seconds=aq_hourly.Interval()),
    inclusive="left"
).tz_convert(aq_response.Timezone().decode()).tz_localize(None)

aq_df = pd.DataFrame({
    "date": aq_dates,
    "pm10": aq_hourly.Variables(0).ValuesAsNumpy(),
    "pm2_5": aq_hourly.Variables(1).ValuesAsNumpy(),
    "carbon_monoxide": aq_hourly.Variables(2).ValuesAsNumpy(),
    "ozone": aq_hourly.Variables(3).ValuesAsNumpy(),
    "nitrogen_dioxide": aq_hourly.Variables(4).ValuesAsNumpy(),
    "sulphur_dioxide": aq_hourly.Variables(5).ValuesAsNumpy(),
    "us_aqi": aq_hourly.Variables(6).ValuesAsNumpy(),
    "us_aqi_pm2_5": aq_hourly.Variables(7).ValuesAsNumpy(),
    "us_aqi_pm10": aq_hourly.Variables(8).ValuesAsNumpy(),
    "us_aqi_nitrogen_dioxide": aq_hourly.Variables(9).ValuesAsNumpy(),
    "us_aqi_carbon_monoxide": aq_hourly.Variables(10).ValuesAsNumpy(),
    "us_aqi_ozone": aq_hourly.Variables(11).ValuesAsNumpy(),
    "us_aqi_sulphur_dioxide": aq_hourly.Variables(12).ValuesAsNumpy(),
    "grass_pollen": aq_hourly.Variables(13).ValuesAsNumpy(),
    "ragweed_pollen": aq_hourly.Variables(14).ValuesAsNumpy(),
    "olive_pollen": aq_hourly.Variables(15).ValuesAsNumpy(),
    "mugwort_pollen": aq_hourly.Variables(16).ValuesAsNumpy(),
    "birch_pollen": aq_hourly.Variables(17).ValuesAsNumpy(),
    "alder_pollen": aq_hourly.Variables(18).ValuesAsNumpy(),
})

# -----------------------------------------------------------
# MERGE BOTH DATASETS
# -----------------------------------------------------------
merged = weather_df.merge(aq_df, on="date", how="inner")

# -----------------------------------------------------------
# ENGINEER METRICS
# -----------------------------------------------------------
merged["planting_readiness"] = (
    (merged["soil_temperature_0cm"].clip(50, 80) - 50) / 30 * 40 +
    (1 - merged["precipitation_probability"] / 100) * 20 +
    (1 - merged["wind_speed_10m"] / 20).clip(0, 1) * 20 +
    (1 - merged["soil_moisture_0_to_1cm"].clip(0, 0.5) / 0.5) * 20
).clip(0, 100)

merged["allergy_risk"] = (
    (merged["pm2_5"] / 35).clip(0, 1) * 25 +
    (merged["ozone"] / 70).clip(0, 1) * 15 +
    (merged[["grass_pollen", "ragweed_pollen", "birch_pollen",
             "alder_pollen", "mugwort_pollen", "olive_pollen"]]
     .fillna(0)
     .mean(axis=1) / 200).clip(0, 1) * 60
).clip(0, 100)

# -----------------------------------------------------------
# LOAD INTO POSTGRESQL
# -----------------------------------------------------------
conn = psycopg2.connect(
   host="aws-1-us-west-2.pooler.supabase.com",
    database="postgres", # Changed from "5432" to "postgres"
    user="postgres.jaluakardtzemqerpdpk",
    password="Un61zM51qN30O3Nw"
)
cur = conn.cursor()

# Drop table if it exists to ensure schema is always correct
cur.execute("DROP TABLE IF EXISTS staging_environmental_raw")
conn.commit()

# Create table if it doesn't exist, to ensure all columns are present
cur.execute("""
CREATE TABLE staging_environmental_raw (
    timestamp_local TIMESTAMP,
    temperature_2m REAL,
    relative_humidity_2m REAL,
    precipitation_probability REAL,
    precipitation REAL,
    wind_speed_10m REAL,
    soil_temperature_0cm REAL,
    soil_moisture_0_to_1cm REAL,
    pm10 REAL,
    pm2_5 REAL,
    carbon_monoxide REAL,
    ozone REAL,
    nitrogen_dioxide REAL,
    sulphur_dioxide REAL,
    us_aqi REAL,
    us_aqi_pm2_5 REAL,
    us_aqi_pm10 REAL,
    us_aqi_nitrogen_dioxide REAL,
    us_aqi_carbon_monoxide REAL,
    us_aqi_ozone REAL,
    us_aqi_sulphur_dioxide REAL,
    grass_pollen REAL,
    ragweed_pollen REAL,
    olive_pollen REAL,
    mugwort_pollen REAL,
    birch_pollen REAL,
    alder_pollen REAL,
    planting_readiness REAL,
    allergy_risk REAL,
    high_pollen_flag BOOLEAN
);
""")
conn.commit()

cur.execute("TRUNCATE staging_environmental_raw")

insert_query = """
INSERT INTO staging_environmental_raw (
    timestamp_local,
    temperature_2m,
    relative_humidity_2m,
    precipitation_probability,
    precipitation,
    wind_speed_10m,
    soil_temperature_0cm,
    soil_moisture_0_to_1cm,
    pm10,
    pm2_5,
    carbon_monoxide,
    ozone,
    nitrogen_dioxide,
    sulphur_dioxide,
    us_aqi,
    us_aqi_pm2_5,
    us_aqi_pm10,
    us_aqi_nitrogen_dioxide,
    us_aqi_carbon_monoxide,
    us_aqi_ozone,
    us_aqi_sulphur_dioxide,
    grass_pollen,
    ragweed_pollen,
    olive_pollen,
    mugwort_pollen,
    birch_pollen,
    alder_pollen,
    planting_readiness,
    allergy_risk,
    high_pollen_flag
)
VALUES %s
"""

records = [
    (
        row["date"],
        row["temperature_2m"],
        row["relative_humidity_2m"],
        row["precipitation_probability"],
        row["precipitation"],
        row["wind_speed_10m"],
        row["soil_temperature_0cm"],
        row["soil_moisture_0_to_1cm"],
        row["pm10"],
        row["pm2_5"],
        row["carbon_monoxide"],
        row["ozone"],
        row["nitrogen_dioxide"],
        row["sulphur_dioxide"],
        row["us_aqi"],
        row["us_aqi_pm2_5"],
        row["us_aqi_pm10"],
        row["us_aqi_nitrogen_dioxide"],
        row["us_aqi_carbon_monoxide"],
        row["us_aqi_ozone"],
        row["us_aqi_sulphur_dioxide"],
        row["grass_pollen"],
        row["ragweed_pollen"],
        row["olive_pollen"],
        row["mugwort_pollen"],
        row["birch_pollen"],
        row["alder_pollen"],
        row["planting_readiness"],
        row["allergy_risk"],
        row["allergy_risk"] > 60
    )
    for _, row in merged.iterrows()
]

execute_values(cur, insert_query, records)
conn.commit()

cur.close()
conn.close()

print("Fresh data loaded into PostgreSQL.")

Fresh data loaded into PostgreSQL.
